In [ ]:
import os

os.listdir('/content')

['.config', 'eng_dataset.csv', 'sample_data']

In [ ]:
import pandas as pd

df = pd.read_csv('/content/eng_dataset.csv')

In [ ]:
df.head()

,ID,sentiment,content
0,10941,anger,At the point today where if someone says somet...
1,10942,anger,@CorningFootball IT'S GAME DAY!!!! T MIN...
2,10943,anger,This game has pissed me off more than any othe...
3,10944,anger,@spamvicious I've just found out it's Candice ...
4,10945,anger,@moocowward @mrsajhargreaves @Melly77 @GaryBar...


In [ ]:
df.columns

Index(['ID', 'sentiment', 'content'], dtype='object')

In [ ]:
# Rename columns to standard names
df.rename(columns={
    'content': 'text',
    'sentiment': 'emotion'
}, inplace=True)

In [ ]:
df[['text', 'emotion']].head()

,text,emotion
0,At the point today where if someone says somet...,anger
1,@CorningFootball IT'S GAME DAY!!!! T MIN...,anger
2,This game has pissed me off more than any othe...,anger
3,@spamvicious I've just found out it's Candice ...,anger
4,@moocowward @mrsajhargreaves @Melly77 @GaryBar...,anger


In [ ]:
df['emotion'].value_counts()

,count
emotion,
fear,2252
anger,1701
joy,1616
sadness,1533


**Transformer**

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import pipeline
import pandas as pd

In [ ]:
emotion_model = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=1
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: SamLowe/roberta-base-go_emotions
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
emotion_model([
    "I am not having a great day",
    "I'm sorry that the order got delayed"
])

[[{'label': 'disappointment', 'score': 0.46669596433639526}],
 [{'label': 'remorse', 'score': 0.761050820350647}]]

In [ ]:
df = pd.read_csv("/content/eng_dataset.csv")

In [ ]:
df.rename(columns={
    "content": "text",
    "sentiment": "emotion"
}, inplace=True)

In [ ]:
df = df[["text", "emotion"]]
df.head()

,text,emotion
0,At the point today where if someone says somet...,anger
1,@CorningFootball IT'S GAME DAY!!!! T MIN...,anger
2,This game has pissed me off more than any othe...,anger
3,@spamvicious I've just found out it's Candice ...,anger
4,@moocowward @mrsajhargreaves @Melly77 @GaryBar...,anger


In [ ]:
sample_df = df.sample(200, random_state=42)
sample_df.shape

(200, 2)

In [ ]:
def get_roberta_emotion(text):
    result = emotion_model(text)
    return result[0][0]['label']

In [ ]:
sample_df["text"].head(5).apply(get_roberta_emotion)

,text
6095,love
4447,neutral
4677,neutral
4832,fear
3796,neutral


In [ ]:
sample_df["roberta_emotion"] = sample_df["text"].apply(get_roberta_emotion)
sample_df.head()

,text,emotion,roberta_emotion
6095,"Trying to loveee somebody, just wanna love som...",joy,love
4447,Andrew's hands start shaking and he says 'I ho...,fear,neutral
4677,Recording some more #FNAF and had to FaceTime ...,fear,neutral
4832,Man Utd are shambles that was horrific ðŸ˜­ðŸ˜...,fear,fear
3796,... flat party and I instantly get bollocked a...,anger,neutral


In [ ]:
import plotly.express as px

fig = px.histogram(
    sample_df,
    x="roberta_emotion",
    color="roberta_emotion",
    title="Emotion Distribution using RoBERTa"
)
fig.show()

In [ ]:
sample_df.to_csv("/content/outputs_roberta.csv", index=False)